# Milestone - Unit 13: evaluating the whole system (ethics and beyond-accuracy) -- Our best path scores about 0.36 hit@10. This milestone asks the questions a hit-rate cannot answer: **who** is the recommender good for, **which books** does it never show, and **what happens** when readers start reading what it suggests? Every claim is a number on our data. The measured story: the most accurate paths are the most **concentrated** (item Gini 0.93 for the two-tower vs 0.77 for item-item CF); a recommender retrained on its own suggestions **narrows** (item-item coverage about 0.49 -> 0.36, Gini about 0.77 -> 0.83 in six rounds); one **exploration slot** buys back part of that exposure; a **popularity penalty** traces an accuracy <-> exposure trade-off curve with no single winner; the offline score itself **reflects the logging policy** (head targets are hit at least 5x as often as tail targets); and the cold-start thread closes: of the learned/collaborative paths only the feature tower ever shows a book nobody has read. These are *descriptive exposure-inequality proxies*, not a complete fairness judgment. Everything is fixed-seed and deterministic, on the frozen `val` split.

## 1. Load the main log and the val cohort

The same generated main interaction log as Units 2-11 (not the Unit-12 session log). The **cohort** is the scoreboard's: readers with at least one unseen `val` positive, the 60 cold readers excluded -- 500 readers. `val_cohort` reproduces that eligibility exactly, and every readout below (the whole-system table, the feedback loops, the trade-off curve) uses those same 500 readers. The **head** is the top 10% of the catalog by train-positive count (200 books), frozen from the original log. A **zero-train** book has no train positive at all -- the cold-start thread's subject.

In [ ]:
import json
from collections import defaultdict
from itertools import pairwise
from time import perf_counter

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    FeatureTowerRetrievalPath,
    HybridRetrievalPath,
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    PopularityRetrievalPath,
    SemanticEmbeddingRetrievalPath,
    TwoTowerRetrievalPath,
    author_catalog_share,
    author_exposure_gini,
    blend,
    catalog_coverage,
    exposure_gini,
    genre_entropy,
    genre_spread_vs_history,
    gini,
    head_ids_from_counts,
    head_share,
    load_glove_subset,
    long_tail_share,
    popularity_penalized,
    rank,
    simulate_feedback_loop,
    val_cohort,
)

K = 10
SEED = 0
notebook_start = perf_counter()

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")
glove = load_glove_subset()
cold = json.loads((data_dir / "cold_partitions.json").read_text())
rows = pd.read_csv(data_dir / "interactions.csv.gz").to_dict("records")

cohort = val_cohort(rows, catalog_ids, cold["cold_readers"])
assert len(cohort) == 500

known = set(catalog_ids)
seen = defaultdict(set)       # each reader's ORIGINAL train positives (never recommended back)
val_positives = defaultdict(set)
for row in rows:
    if int(row["label"]) != 1 or row["item_id"] not in known:
        continue
    if row["split"] == "train":
        seen[row["reader_id"]].add(row["item_id"])
    elif row["split"] == "val":
        val_positives[row["reader_id"]].add(row["item_id"])
relevant = {reader: val_positives[reader] - seen[reader] for reader in cohort}

{
    "books": len(catalog_ids),
    "interactions": len(rows),
    "val_cohort": len(cohort),
}

## 2. Fit the paths of Units 2-10

The cheap paths fit in well under a second: Unit-2 **popularity**, Unit-3 **lexical** (BM25), Unit-4 **item-item** CF and Unit-7 **semantic** (GloVe). Three fits cost real time, each at its pinned config and seed and each in its own cell: Unit-5 **MF**, the Unit-8 **two-tower** and the Unit-9 **feature tower**. The Unit-10 **hybrid** and the Unit-6 **blend** reuse those fits, so they cost nothing extra. The popularity path's `counts` (train-positive count per book) define the head and the zero-train set.

In [ ]:
fit_start = perf_counter()
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
semantic = SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)
mf = MatrixFactorizationPath(seed=SEED).fit(rows, catalog=catalog_ids)
mf_seconds = perf_counter() - fit_start

counts = popularity.counts                                 # {book: train-positive count}
head = set(head_ids_from_counts(counts, catalog_ids))     # top 10% by train count
zero_train = set(catalog_ids) - set(counts)               # books with no train positive at all

print(f"popularity, lexical, item-item, semantic + MF fits: {mf_seconds:.1f}s")
{"head_books": len(head), "zero_train_books": len(zero_train)}

In [ ]:
two_tower_start = perf_counter()
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)   # Unit 8, ID-only towers
two_tower_seconds = perf_counter() - two_tower_start
print(f"two-tower fit: {two_tower_seconds:.1f}s")

In [ ]:
feature_tower_start = perf_counter()
# Unit 9's pinned config: id + genre + author + projected GloVe item features, warm-only negatives.
feature_tower = FeatureTowerRetrievalPath(
    catalog, keywords, glove, n_epochs=40, negative_pool="warm", seed=SEED
).fit(rows, catalog=catalog_ids)
feature_tower_seconds = perf_counter() - feature_tower_start
print(f"feature-tower fit: {feature_tower_seconds:.1f}s")

The two composite paths. The **U10 hybrid** is Unit 10's shipped `HybridRetrievalPath` (BM25 + two-tower, weight 0.7, pool 50). The **U6 blend** is Unit 6's first blended recommender, reused **exactly**: every member retrieves a pool of 30, the scores are min-max calibrated and summed with the pinned weights {popularity 0.25, lexical 0.5, item-item 1.0, mf 1.0}, and the reader's seen books are dropped before the top-k. Wrapping it in a tiny `retrieve` class lets it sit in every table below (and inside the popularity-penalty re-rank) like any other path.

In [ ]:
hybrid = HybridRetrievalPath(lexical, two_tower, weight=0.7, pool=50, method="weighted").fit(rows)

U6_POOL = 30
U6_WEIGHTS = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}
u6_members = {"popularity": popularity, "lexical": lexical, "item-item": item_item, "mf": mf}


class U6Blend:
    # Unit 6's pinned blend: pool 30 per member, calibrated weighted union, seen books dropped.
    # The answer for one (reader, k, seen set) never changes, so it is computed once and reused
    # (the lambda grid in section 6 asks the same question five times).
    name = "u6-blend"

    def __init__(self):
        self.memo = {}

    def retrieve(self, query, context, k):
        already = frozenset(context.get("seen", ()))
        key = (query, k, already)
        if key not in self.memo:
            per_path = {name: path.retrieve(query, context, U6_POOL) for name, path in u6_members.items()}
            self.memo[key] = [c for c in blend(per_path, weights=U6_WEIGHTS) if c.item_id not in already][:k]
        return self.memo[key]


u6_blend = U6Blend()
{"u6_pool": U6_POOL, "u6_weights": U6_WEIGHTS}

## 3. The whole-system scoreboard: accuracy *and* who gets shown

For each path, every cohort reader's top-10 on `val`, then six readouts:
- **hit@10** -- the accuracy we have optimized all book long;
- **coverage** -- the fraction of the 2000-book catalog shown to at least one reader;
- **head-share** -- the fraction of all recommendation slots that go to head books (the long-tail share is 1 minus it);
- **item Gini** -- the Gini coefficient of per-book exposure (slot counts) over the **full catalog**, never-shown books counted as 0, so every path is measured over the same N (0 = every book shown equally, near 1 = a few books take every slot);
- **author Gini** -- the same at the author level;
- **genre H** -- the mean Shannon entropy (bits) of the genres in each reader's list (how wide a slice of genres a reader is shown).

In [ ]:
def top_lists(path, ask=K):
    # Each cohort reader's top-K ids on val (seen books excluded), in cohort order.
    return {
        reader: [c.item_id for c in rank(path.retrieve(reader, {"seen": seen[reader]}, ask), n=K, exclude=seen[reader])]
        for reader in cohort
    }


def hit_at_k(lists):
    return float(np.mean([1.0 if set(top) & relevant[reader] else 0.0 for reader, top in lists.items()]))


def whole_system_row(name, lists):
    return {
        "path": name,
        "hit@10": round(hit_at_k(lists), 4),
        "coverage": round(catalog_coverage(lists.values(), catalog_ids), 4),
        "head-share": round(head_share(lists.values(), head), 4),
        "item Gini": round(exposure_gini(lists, catalog_ids), 4),
        "author Gini": round(author_exposure_gini(lists, catalog), 4),
        "genre H": round(float(np.mean([genre_entropy(top, catalog) for top in lists.values()])), 3),
    }


paths = {
    "popularity": popularity,
    "lexical": lexical,
    "semantic": semantic,
    "item-item": item_item,
    "MF": mf,
    "two-tower": two_tower,
    "feature tower": feature_tower,
    "U10 hybrid": hybrid,
    "U6 blend": u6_blend,
}
table_start = perf_counter()
served = {name: top_lists(path) for name, path in paths.items()}
whole_system_rows = [whole_system_row(name, lists) for name, lists in served.items()]
table_seconds = perf_counter() - table_start

# The U6 blend is reused exactly: it reproduces Unit 6's pinned val readout (hit 0.306, coverage 0.333).
u6_row = next(row for row in whole_system_rows if row["path"] == "U6 blend")
assert abs(u6_row["hit@10"] - 0.306) < 0.001 and abs(u6_row["coverage"] - 0.333) < 0.001
# Long-tail share is just 1 - head-share (the helper agrees with the table).
assert abs(long_tail_share(served["item-item"], head) - (1 - head_share(served["item-item"].values(), head))) < 1e-12

print(f"top-10 lists for {len(paths)} paths x {len(cohort)} readers: {table_seconds:.1f}s")
pd.DataFrame(whole_system_rows)

**Reading the author column fairly.** Authors do not own equal numbers of books, so equal exposure per *author* is not the right zero point. `author_catalog_share` gives each author's share of the catalog; the Gini of those shares is the author Gini a recommender would have if it exposed every author **in proportion to their books**. An author Gini near that value is "exposure follows the catalog"; well above it is concentration beyond what the catalog explains.

In [ ]:
catalog_share = author_catalog_share(catalog)
catalog_author_gini = gini(catalog_share.values())
print(f"authors: {len(catalog_share)} | largest author's catalog share: {max(catalog_share.values()):.4f}")
print(f"author Gini of the CATALOG itself (exposure proportional to books): {catalog_author_gini:.4f}")
pd.DataFrame([
    {"path": row["path"], "hit@10": row["hit@10"], "author Gini": row["author Gini"],
     "author Gini - catalog": round(row["author Gini"] - catalog_author_gini, 4)}
    for row in whole_system_rows
])

The story in the table: **the most accurate paths are the most concentrated.** The two-tower (hit about 0.34) and the U10 hybrid (about 0.36) give roughly 90% and 86% of all slots to the 200 head books, with item Gini about 0.93 and 0.92; MF is even narrower (about 7% coverage). Item-item CF is the least concentrated collaborative path (coverage 0.49, Gini 0.77) at a hit of about 0.25. Popularity shows the same 16 books to everyone (Gini 0.99). The content paths (lexical, semantic) spread exposure but are weak on accuracy. No path is best at both -- which is a trade-off to *choose*, not a bug to fix.

## 4. Feedback loops: a recommender retrained on its own suggestions

`simulate_feedback_loop` runs a *stylized exposure-to-consumption loop*. It is **not** a realistic model of what readers like -- readers simply take what they are shown with a fixed probability -- but it isolates one mechanism. Each round:
1. fit the path on the **augmented** train log (the original train rows plus every simulated read so far);
2. serve each cohort reader a top-10, excluding everything they have already read (original + simulated);
3. each served book is read independently with probability p = 0.3 -- **unless** it is one of the reader's own `val`/`test` books (the **leakage guard**: the simulation must never hand the model a held-out answer, or the accuracy readout would rise for a fake reason);
4. append the reads as new train rows, and refit next round.

Round 0 is the path on the original log; we run R = 6 more rounds with seed 0. Each round reports the coverage, head-share (head frozen at round 0), item Gini and genre H of the **served** lists, and a **clean hit@10**: the refit path scored on the original `val` books with the original seen set, so the accuracy column moves only because the *model* changed.

Two loops: item-item CF (the least concentrated collaborative path) and popularity (already maximally concentrated).

In [ ]:
loop_start = perf_counter()
item_item_loop = simulate_feedback_loop(ItemItemRetrievalPath, rows, cohort=cohort, catalog=catalog, seed=SEED)
item_item_loop_seconds = perf_counter() - loop_start
popularity_start = perf_counter()
popularity_loop = simulate_feedback_loop(PopularityRetrievalPath, rows, cohort=cohort, catalog=catalog, seed=SEED)
popularity_loop_seconds = perf_counter() - popularity_start


def loop_rows(label, results):
    return [
        {
            "loop": label,
            "round": m.round,
            "coverage": round(m.coverage, 4),
            "head-share": round(m.head_share, 4),
            "item Gini": round(m.gini, 4),
            "genre H": round(m.genre_entropy, 3),
            "clean hit@10": round(m.clean_hit, 4),
            "reads appended": m.accepted,
        }
        for m in results
    ]


first, last = item_item_loop[0], item_item_loop[-1]
assert len(item_item_loop) == 7 and first.round == 0 and last.round == 6
# GATE (a): the item-item loop concentrates -- coverage falls by >= 0.08 and the Gini rises by >= 0.03.
assert last.coverage <= first.coverage - 0.08
assert last.gini >= first.gini + 0.03
# GATE (a): the popularity loop stays maximally concentrated in every round.
assert all(m.gini >= 0.98 and m.coverage <= 0.03 for m in popularity_loop)

print(f"item-item loop (7 fits): {item_item_loop_seconds:.1f}s | popularity loop: {popularity_loop_seconds:.1f}s")
print(f"item-item: coverage {first.coverage:.3f} -> {last.coverage:.3f}, Gini {first.gini:.4f} -> {last.gini:.4f}, "
      f"clean hit {first.clean_hit:.3f} -> {last.clean_hit:.3f}")
pd.DataFrame(loop_rows("item-item", item_item_loop) + loop_rows("popularity", popularity_loop))

**Rich get richer.** Each round, the books item-item CF already likes get read more, so their co-occurrence counts grow, so they get shown more. Coverage shrinks and the Gini climbs, round after round. The clean hit falls too: retraining on its own suggestions teaches CF the *recommender's* taste, not the readers'. Popularity cannot get much more concentrated than it starts -- it is the extreme case from round 0.

### One mitigation: an exploration slot

Replace the **last** of the ten slots with a seeded random warm book (one with at least one original train read) that the reader has not seen and that is not already in the list. Same seed, same loop, same guard.

In [ ]:
explore_start = perf_counter()
explore_loop = simulate_feedback_loop(
    ItemItemRetrievalPath, rows, cohort=cohort, catalog=catalog, seed=SEED, mitigation="explore"
)
explore_loop_seconds = perf_counter() - explore_start

loop_last, explore_last = item_item_loop[-1], explore_loop[-1]
# GATE (b): at round R the exploration slot restores exposure vs the unmitigated loop.
assert explore_last.coverage >= loop_last.coverage + 0.04
assert explore_last.gini <= loop_last.gini - 0.01

# The accuracy cost is REPORTED, not gated: two signed deltas (mitigated - unmitigated).
cost_0 = explore_loop[0].clean_hit - item_item_loop[0].clean_hit
cost_R = explore_last.clean_hit - loop_last.clean_hit
print(f"exploration loop (7 fits): {explore_loop_seconds:.1f}s")
print(f"round {loop_last.round}: coverage {loop_last.coverage:.4f} -> {explore_last.coverage:.4f} "
      f"({explore_last.coverage - loop_last.coverage:+.4f}), Gini {loop_last.gini:.4f} -> {explore_last.gini:.4f} "
      f"({explore_last.gini - loop_last.gini:+.4f})")
print(f"clean-hit cost (reported, not gated): cost_0 = {cost_0:+.4f}, cost_R = {cost_R:+.4f}")
pd.DataFrame(loop_rows("item-item + explore", explore_loop))

The exploration slot buys back part of the lost exposure (more coverage, a lower Gini) at round R. What it costs depends on when you look: at round 0 the clean hit is identical (`cost_0` = 0 -- the clean readout scores the refit path, and round 0 fits the same original log), and by round R the signed `cost_R` above is what six rounds of different reads did to the model. On the committed seed it is *positive* (about +0.05): the unmitigated loop's clean hit decays faster (0.252 -> 0.090) than the explored loop's (0.252 -> 0.142), so here the slot costs nothing measurable -- it slows the decay. It narrows the loop's damage; it does not undo it.

## 5. Filter bubbles: is the reader shown a narrower slice of genres?

Two readouts. First, **per reader**: `genre_spread_vs_history` compares the genre entropy of each reader's top-10 with the genre entropy of the books they actually read. Second, **over the loop**: the mean genre H of the served lists, round by round.

In [ ]:
spread_rows = []
for name in ("popularity", "item-item", "MF", "two-tower", "U6 blend"):
    spread = genre_spread_vs_history(served[name], {reader: seen[reader] for reader in cohort}, catalog)
    recommended = np.array([pair[0] for pair in spread.values()])
    history = np.array([pair[1] for pair in spread.values()])
    spread_rows.append({
        "path": name,
        "mean genre H of top-10": round(float(recommended.mean()), 3),
        "mean genre H of history": round(float(history.mean()), 3),
        "readers shown a narrower slice": round(float((recommended < history).mean()), 3),
    })

entropy = [m.genre_entropy for m in item_item_loop]
dip_round = min(range(1, 5), key=lambda r: entropy[r])
dip = entropy[dip_round]
# GATE (d): the item-item loop narrows the genre spread by at least 0.3 bits within rounds 1-4.
assert dip <= entropy[0] - 0.3
print(f"item-item loop genre H: round 0 {entropy[0]:.3f} -> min(rounds 1-4) {dip:.3f} "
      f"(round {dip_round}) -> round 6 {entropy[-1]:.3f} (rebound {entropy[-1] - dip:+.3f})")
pd.DataFrame(spread_rows)

The narrowing is real but **transient**: the genre spread drops by more than 0.3 bits within a few rounds (1.64 -> about 1.18 bits by round 4), then partly rebounds by round 6 even while coverage keeps falling -- fewer books, but not fewer genres. A single number ("the bubble is X bits") would hide that shape, so we report the dip *and* the rebound.

## 6. Exposure fairness as a trade-off curve: the popularity penalty

`popularity_penalized` re-ranks a path's top-50 pool: `score' = minmax(score) - lambda * log1p(count) / log1p(max count)`. A larger lambda pushes popular books down. The grid is predeclared -- lambda in {0, 0.1, 0.3, 0.5, 1.0} -- and we run it on item-item CF and on the U6 blend. This is a **Pareto readout**: every point is a legitimate operating choice, and *which* one to ship is a judgment about who the system should serve, not something the curve decides.

In [ ]:
LAMBDAS = (0.0, 0.1, 0.3, 0.5, 1.0)
ANCHORS = {"item-item": (0.224, 0.747), "U6 blend": (0.290, 0.858)}   # lambda = 0.3 (hit, Gini)

curve_start = perf_counter()
curve_rows = []
for base_name, base in (("item-item", item_item), ("U6 blend", u6_blend)):
    hits, ginis = [], []
    for lam in LAMBDAS:
        lists = top_lists(popularity_penalized(base, lam, counts, pool=50))
        hits.append(hit_at_k(lists))
        ginis.append(exposure_gini(lists, catalog_ids))
        curve_rows.append({
            "base": base_name,
            "lambda": lam,
            "hit@10": round(hits[-1], 4),
            "hit delta vs lambda 0": round(hits[-1] - hits[0], 4),
            "item Gini": round(ginis[-1], 4),
            "coverage": round(catalog_coverage(lists.values(), catalog_ids), 4),
            "head-share": round(head_share(lists.values(), head), 4),
        })
    # GATE (c): over the grid the Gini strictly decreases and the hit never increases.
    assert all(a > b for a, b in pairwise(ginis)), (base_name, ginis)
    assert all(a >= b for a, b in pairwise(hits)), (base_name, hits)
    # GATE (c) anchors at lambda = 0.3 (+-0.003 hit, +-0.005 Gini).
    anchor_hit, anchor_gini = ANCHORS[base_name]
    assert abs(hits[LAMBDAS.index(0.3)] - anchor_hit) <= 0.003, (base_name, hits)
    assert abs(ginis[LAMBDAS.index(0.3)] - anchor_gini) <= 0.005, (base_name, ginis)
curve_seconds = perf_counter() - curve_start

print(f"penalty grid ({len(LAMBDAS)} lambdas x 2 bases): {curve_seconds:.1f}s")
pd.DataFrame(curve_rows)

Both curves slope the same way: every step of lambda buys a lower Gini and costs some hit@10. On item-item, lambda = 0.3 gives up about 0.03 hit for about 0.03 Gini; on the U6 blend, about 0.016 hit for about 0.05 Gini; lambda = 1.0 costs far more. Coverage is **not** a gated column: it is non-monotone in lambda (item-item's peaks around lambda = 0.5), because pushing the head down eventually concentrates exposure on a *different* set of books. Who decides the penalty -- and on whose behalf -- is the real question.

## 7. Offline-metric limits: the score reflects the logging policy

Our `val` positives were logged by readers who mostly found popular books, so the held-out targets themselves lean to the head. Split each reader's unseen val books into **head targets** and **tail targets** and score hit@10 on each separately. This is a descriptive measurement of what the offline number rewards -- not a causal claim that the metric is inflated.

In [ ]:
positives = [book for reader in cohort for book in relevant[reader]]
head_target_share = float(np.mean([book in head for book in positives]))
assert abs(head_target_share - 0.441) < 0.005

head_tail_rows = []
for name, lists in served.items():
    head_hits, tail_hits = [], []
    for reader, top in lists.items():
        head_targets, tail_targets = relevant[reader] & head, relevant[reader] - head
        if head_targets:
            head_hits.append(1.0 if set(top) & head_targets else 0.0)
        if tail_targets:
            tail_hits.append(1.0 if set(top) & tail_targets else 0.0)
    head_tail_rows.append({
        "path": name,
        "head-target hit@10": round(float(np.mean(head_hits)), 4),
        "tail-target hit@10": round(float(np.mean(tail_hits)), 4),
        "head readers": len(head_hits),
        "tail readers": len(tail_hits),
    })
head_tail = {row["path"]: row for row in head_tail_rows}

# GATE (d): for the collaborative paths and the U6 blend, head targets are hit >= 5x as often (tail may be 0).
for name in ("item-item", "MF", "two-tower", "U6 blend"):
    assert head_tail[name]["head-target hit@10"] >= 5 * head_tail[name]["tail-target hit@10"], name

print(f"share of the cohort's unseen val positives that are head books: {head_target_share:.3f}")
pd.DataFrame(head_tail_rows)

The collaborative paths hit head targets many times as often as tail targets (MF essentially never hits a tail target), while the content paths are far more even. A path that only ever recommends head books can still score well offline, because 44% of the targets *are* head books -- the logged data rewards the same popularity the logging policy produced. Offline hit@10 is a measurement of agreement with past behaviour, not of what readers would have liked had they been shown more.

## 8. The cold-start thread closes

Unit 6 found that warm-only paths cannot serve a book nobody has read; Unit 9 built the feature tower to fix that. Here is the final readout across every path: how many distinct **zero-train** books (no train positive at all) appear anywhere in the cohort's top-10 lists.

In [ ]:
cold_rows = []
for name, lists in served.items():
    surfaced = {book for top in lists.values() for book in top} & zero_train
    cold_rows.append({"path": name, "zero-train books surfaced": len(surfaced),
                      "share of zero-train books": round(len(surfaced) / len(zero_train), 4)})
cold_readout = {row["path"]: row["zero-train books surfaced"] for row in cold_rows}

# The warm-only collaborative paths surface exactly 0 zero-train books; the feature tower surfaces some.
for name in ("popularity", "item-item", "MF", "two-tower"):
    assert cold_readout[name] == 0, name
assert cold_readout["feature tower"] > 0

total_seconds = perf_counter() - notebook_start
print(f"zero-train books in the catalog: {len(zero_train)} | whole notebook so far: {total_seconds:.1f}s")
pd.DataFrame(cold_rows)

**The honest headline.** Accuracy is one column of a wider table. On our data the most accurate paths concentrate exposure on a few head books; a recommender retrained on its own suggestions narrows further (item-item coverage about 0.49 -> 0.36, Gini about 0.77 -> 0.83, and a genre-spread dip of more than 0.3 bits); an exploration slot and a popularity penalty each buy back exposure at a measured accuracy cost, and choosing the point on that curve is a decision about who the system serves; the offline score itself rewards the head because the logged targets lean to it; and among the learned/collaborative paths only the feature tower ever shows a book nobody has read (the content paths, lexical and semantic, surface even more). Every number here is a descriptive proxy on a stylized simulation and a synthetic log -- evidence for a trade-off discussion, not a verdict.

**The bridge.** Checkpoint B assesses Part 2 and finally opens the sealed `test` holdout -- once, after freezing on `val`; Unit 14's capstone wires retrieval -> blend -> rerank into one system, and this unit's whole-system table is how it will be judged.